# Model Training

Objective:
Build a machine learning pipeline capable of predicting SLA breaches at incident creation time.

The modeling phase includes:

- Preprocessing Pipeline
- Missing Value Handling
- Encoding
- Class Imbalance Handling
- Model Training
- Cross Validation
- Hyperparameter Tuning
- Model Selection

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,TargetEncoder,FunctionTransformer,BusinessMissingValueTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,roc_auc_score



# Load Processed Data

In [18]:
X_train=pd.read_csv('../../data/processed/X_train.csv')
X_test=pd.read_csv('../../data/processed/X_test.csv')
y_train=pd.read_csv('../../data/processed/y_train.csv')
y_test=pd.read_csv('../../data/processed/y_test.csv')

# Feature Groups

| Feature Group      | Features                                                        |
| ------------------ | --------------------------------------------------------------- |
| Frequency Encoding | category, subcategory, assignment_group, assigned_to, opened_by |
| Target Encoding    | caller_id, location, u_symptom, contact_type                    |
| Ordinal Encoding   | impact, urgency, priority                                       |
| Numerical          | Hour                                                            |
| Calendar Features  | Month, Day_of_week                                              |
                                          |


## Preprocessing Pipeline Design

### Traditional ML Pipeline

Missing Value Replacement
        ↓
Ordinal Encoding
        ↓
Frequency Encoding
        ↓
Target Encoding
        ↓
Model

### CatBoost Pipeline

Missing Value Replacement
        ↓
Ordinal Encoding
        ↓
Native Categorical Handling
        ↓
CatBoost

# Custom Transformers

In [42]:

replacement={
    'location': 'Unknown_location',
    'category':'Unknown_category',
    'subcategory':'Unknown_subcategory',
    'u_symptom':'Unknown_symptom',
    'assignment_group':'Unknown_assignment_group',
    'assigned_to':'Unknown_assigned_to',
    'caller_id': 'Unknown_caller_id',
    'opened_by':'Unknown_opened_by'
}

nested_replacement={col: {'?':val} for col, val in replacement.items()}

def replace_missing_value(X):
    X_copy=X.copy()
    return X_copy.replace(nested_replacement)

missing_value_imputer = FunctionTransformer(replace_missing_value)

missing_values_pipeline=Pipeline([('impute_missing_value',missing_value_imputer)])



In [43]:
X_train_clean=missing_values_pipeline.fit_transform(X_train)

In [50]:
(X_train_clean=='Unknown_location').sum()
# X_train.shape,X_train_clean.shape

contact_type         0
location            56
category             0
subcategory          0
u_symptom            0
assignment_group     0
assigned_to          0
caller_id            0
opened_by            0
impact               0
urgency              0
priority             0
Hour                 0
Day_of_week          0
Month                0
dtype: int64

## Modeling Strategy

Baseline Models
- Logistic Regression
- Decision Tree
- Random Forest

Boosting Models
- XGBoost
- LightGBM
- CatBoost

Evaluation Metrics
- Precision
- Recall
- F1 Score
- ROC-AUC
- PR-AUC

Cross Validation
- Stratified K-Fold

Hyperparameter Optimization
- Optuna